# EU Competition Cases - Load & Explore JSON

Load the merged JSON dataset into one pandas DataFrame, or rebuild that DataFrame from the 5 source JSON files (AT, DMA, FS, M, SA), then run basic analysis to inform the knowledge-graph schema.

In [ ]:
import json, re
import pandas as pd
from collections import Counter, defaultdict
from pathlib import Path

DATA_DIR = Path("Data")
MERGED_PATH = DATA_DIR / "cases_merged.json"
source_files = sorted(DATA_DIR.glob("case-data-*.json"))

print(f"JSON dataset: {MERGED_PATH}")
print("Fallback source JSON files:")
for path in source_files:
    print(f"  {path.name:25s} {path.stat().st_size/1_000_000:7.2f} MB")

In [ ]:
# --- Load JSON data into one DataFrame. Prefer the merged JSON file; rebuild if missing. ---
def load_source_json_frame(files):
    records = []
    seen_case_ids = {}
    collisions = []

    for path in files:
        instrument = path.stem.replace("case-data-", "")  # AT, DMA, FS, M, SA
        with open(path, encoding="utf-8") as f:
            data = json.load(f)
        for case_id, case in data.items():
            if case_id in seen_case_ids:
                collisions.append((case_id, seen_case_ids[case_id], instrument))
            seen_case_ids[case_id] = instrument
            records.append({"case_id": case_id, **case, "_sourceFile": instrument})

    return pd.DataFrame.from_records(records), collisions


if MERGED_PATH.exists():
    with open(MERGED_PATH, encoding="utf-8") as f:
        data = json.load(f)

    df = pd.DataFrame.from_dict(data, orient="index").reset_index(names="case_id")
    collisions = []
    print(f"Loaded {MERGED_PATH} into one DataFrame ({MERGED_PATH.stat().st_size/1_000_000:.2f} MB)")
else:
    df, collisions = load_source_json_frame(source_files)
    print("Merged JSON not found; loaded source JSON files into one DataFrame instead.")

if "_sourceFile" not in df.columns:
    df["_sourceFile"] = "unknown"

per_file_counts = df["_sourceFile"].fillna("unknown").value_counts().sort_index().to_dict()
print(f"Cases per source: {per_file_counts}")
print(f"Total cases: {len(df)}")
print(f"DataFrame shape: {df.shape}")
print(f"ID collisions across source files: {len(collisions)}")
if collisions[:5]:
    print("First 5 collisions:", collisions[:5])

df.head()

In [ ]:
# --- Schema scan: which metadata fields exist, and how often ---
field_counts = Counter()
field_per_instrument = defaultdict(Counter)

for _, row in df.iterrows():
    inst = row.get("_sourceFile", "unknown")
    metadata = row.get("metadata") or {}
    for k in metadata.keys():
        field_counts[k] += 1
        field_per_instrument[inst][k] += 1

print(f"{'field':45s} {'overall':>8s}  per-instrument coverage")
print("-" * 100)
for field, n in field_counts.most_common():
    coverage = " ".join(
        f"{inst}:{field_per_instrument[inst][field]}/{per_file_counts[inst]}"
        for inst in per_file_counts
    )
    print(f"{field:45s} {n:>8d}  {coverage}")

In [ ]:
# --- Decisions & attachments: how nested is the data? ---
decisions_per_case = []
attachments_per_decision = []
attachments_per_case = []
case_attachments_per_case = []

for _, row in df.iterrows():
    decs = row.get("decisions") or []
    decisions_per_case.append(len(decs))
    n_att = 0
    for d in decs:
        a = len(d.get("decisionAttachments", []) or [])
        attachments_per_decision.append(a)
        n_att += a
    attachments_per_case.append(n_att)
    case_attachments_per_case.append(len(row.get("caseAttachments") or []))

def summarize(name, xs):
    if not xs:
        print(f"{name}: (empty)"); return
    xs_sorted = sorted(xs)
    print(f"{name:35s} n={len(xs)}  min={min(xs)}  max={max(xs)}  "
          f"mean={sum(xs)/len(xs):.2f}  median={xs_sorted[len(xs)//2]}  "
          f"sum={sum(xs)}")

summarize("decisions per case",        decisions_per_case)
summarize("attachments per decision",  attachments_per_decision)
summarize("attachments per case",      attachments_per_case)
summarize("case-level attachments",    case_attachments_per_case)
print()
print("Decisions-per-case distribution:", dict(Counter(decisions_per_case).most_common()))

In [ ]:
# --- Helpers: many metadata values are lists; some are JSON-encoded strings ---
def first(v):
    """Return first element of a list-valued metadata field, or None."""
    if isinstance(v, list) and v:
        return v[0]
    return v

def parse_embedded(v):
    """Some fields store JSON as a string, e.g. caseSectors, casePressReleases.
    Returns a list of dicts, or [] on failure."""
    s = first(v)
    if not isinstance(s, str):
        return []
    try:
        obj = json.loads(s)
        return obj.get("items", []) if isinstance(obj, dict) else []
    except json.JSONDecodeError:
        return []

# Sanity check on one case
sample = df.iloc[0]
sample_id = sample["case_id"]
print("Sample case:", sample_id)
md = sample["metadata"]
print("  title    :", first(md.get("caseTitle")))
print("  dg       :", first(md.get("caseDg")))
print("  init date:", first(md.get("caseInitiationDate")))
print("  sectors  :", parse_embedded(md.get("caseSectors")))

In [ ]:
# --- Top companies, sectors, DGs, legal bases ---
companies = Counter()
sectors   = Counter()
dgs       = Counter()
legal     = Counter()
case_types = Counter()
languages  = Counter()

for md in df["metadata"]:
    md = md or {}
    for c in md.get("caseCompanies", []) or []:
        companies[c.strip()] += 1
    for s in parse_embedded(md.get("caseSectors")):
        if s.get("label"):
            sectors[s["label"]] += 1
    if (d := first(md.get("caseDg"))):       dgs[d] += 1
    for lb in md.get("caseLegalBasis", []) or []:
        try:
            legal[json.loads(lb).get("label", lb)] += 1
        except Exception:
            legal[lb] += 1
    if (t := first(md.get("caseType"))):     case_types[t] += 1
    if (l := first(md.get("language"))):     languages[l] += 1

def show(title, c, n=10):
    print(f"\n{title} (top {n} of {len(c)} distinct)")
    for k, v in c.most_common(n):
        print(f"  {v:5d}  {k}")

show("Companies", companies)
show("Sectors", sectors)
show("DGs", dgs, n=5)
show("Legal bases", legal, n=10)
show("Case types", case_types, n=10)
show("Languages", languages, n=5)

In [ ]:
# --- Time coverage: initiation year distribution by instrument ---
year_re = re.compile(r"(\d{4})")
by_year = defaultdict(Counter)  # year -> {instrument: count}

for _, row in df.iterrows():
    inst = row.get("_sourceFile", "unknown")
    raw = first((row.get("metadata") or {}).get("caseInitiationDate"))
    if not raw:
        continue
    m = year_re.search(raw)
    if m:
        by_year[int(m.group(1))][inst] += 1

if by_year:
    years = sorted(by_year)
    insts = sorted(per_file_counts)
    print(f"{'year':>6s} " + " ".join(f"{i:>5s}" for i in insts) + "  total")
    for y in years:
        row = by_year[y]
        total = sum(row.values())
        print(f"{y:>6d} " + " ".join(f"{row.get(i,0):>5d}" for i in insts) + f"  {total:>5d}")
    print(f"\nRange: {years[0]} – {years[-1]}")
else:
    print("No initiation dates found.")

In [ ]:
# --- Decision-level analysis: types and adoption-year distribution ---
dec_types = Counter()
dec_years = Counter()
att_categories = Counter()
att_languages = Counter()

for decisions in df["decisions"]:
    for d in decisions or []:
        dmd = d.get("metadata", {})
        for t in dmd.get("decisionTypes", []) or []:
            try:
                dec_types[json.loads(t).get("label", t)] += 1
            except Exception:
                dec_types[t] += 1
        raw = first(dmd.get("decisionAdoptionDate"))
        if raw and (m := year_re.search(raw)):
            dec_years[int(m.group(1))] += 1
        for att in d.get("decisionAttachments", []) or []:
            amd = att.get("metadata", {})
            for c in amd.get("attachmentCategory", []) or []:
                try:
                    att_categories[json.loads(c).get("label", c)] += 1
                except Exception:
                    att_categories[c] += 1
            if (lang := first(amd.get("attachmentLanguage"))):
                att_languages[lang] += 1

show("Decision types", dec_types, n=15)
show("Attachment categories", att_categories, n=15)
show("Attachment languages", att_languages, n=10)

if dec_years:
    print("\nDecisions adopted per year:")
    for y in sorted(dec_years):
        print(f"  {y}: {dec_years[y]}")

In [ ]:
# --- Save the one-DataFrame dataset for the KG-building step ---
out_path = DATA_DIR / "cases_merged.json"
df.set_index("case_id").to_json(out_path, orient="index", force_ascii=False, indent=2)
print(f"Wrote {out_path}  ({out_path.stat().st_size/1_000_000:.2f} MB, {len(df)} cases)")